In [1]:
import cv2 
import mediapipe as mp 
import time 

2025-01-25 13:00:32.950259: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-25 13:00:32.954751: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-25 13:00:32.970901: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1737790232.998690    5930 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1737790233.006498    5930 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-25 13:00:33.033846: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

In [2]:
class handtracking: 
    def __init__(self, static_image_mode = False, max_num_hands = 2, min_detection_confidence = 0.5, min_tracking_confidence = 0.5):
        self.static_image_mode = static_image_mode 
        self.max_num_hands = max_num_hands 
        self.min_detection_confidence = min_detection_confidence
        self.min_tracking_confidence = min_tracking_confidence

        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(static_image_mode = self.static_image_mode, max_num_hands = self.max_num_hands, min_detection_confidence = self.min_detection_confidence, min_tracking_confidence = self.min_tracking_confidence) 

        self.mp_draw = mp.solutions.drawing_utils

    def draw_lanmarks(self, handLms, img, id, draw = True): 
        for ids, lm in enumerate(handLms.landmark):
            h, w, c = img.shape 
            cx, cy = int(lm.x * w), int(lm.y * h) 
            if draw:
                if ids == id:
                    cv2.circle(img, (cx, cy), 15, (255, 0, 255), cv2.FILLED)

    
    def find_hand(self, img, draw = True): 
        imgRgb = cv2.cvtColor(src=img, code=cv2.COLOR_BGR2RGB) 
        results = self.hands.process(imgRgb) 
        if results.multi_hand_landmarks: 
            for handLms in results.multi_hand_landmarks:
                self.draw_lanmarks(handLms, img, 4, draw=True) 
                if draw:
                    self.mp_draw.draw_landmarks(img, handLms, self.mp_hands.HAND_CONNECTIONS)

In [3]:
def main(): 
    pTime = 0
    cTime = 0 
    cap = cv2.VideoCapture(0) 
    if not cap.isOpened(): 
        print("can not open the camera") 
        exit() 
    ht = handtracking(static_image_mode=False, max_num_hands=2, min_detection_confidence=0.5, min_tracking_confidence=0.5) 

    while True: 
        success, img = cap.read() 
        if not success: 
            print("can not read the image") 
            break 
        cTime = time.time() 
        fps = 1 / (cTime- pTime) 
        pTime = cTime 
        ht.find_hand(img=img, draw=True)
        cv2.putText(img=img, text=str(int(fps)), org=(10, 70), fontFace=cv2.FONT_HERSHEY_COMPLEX, fontScale=1, color=(255, 0, 0), thickness=2) 
        cv2.imshow(winname="IMAGE", mat=img) 
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break 

    cap.release() 
    cv2.destroyAllWindows()  


In [ ]:
if __name__ == "__main__": 
    main()